In [194]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import os
import requests
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [195]:
uri = os.getenv('SBS_V1_MONGO_URI')

client = MongoClient(uri, server_api=ServerApi('1'))
db = client['SBSV1']

try:
    client.admin.command('ping')
    print('Pinged your deployment. You successfully connected to MongoDB!')
except Exception as e:
    print(e)

nba_games_historical_collection = db['nba_games_historical']
nba_team_aggregated_game_stats_historical_collection = db['nba_team_aggregated_game_stats_historical']
nba_game_player_stats_historical_collection = db['nba_game_player_stats_historical']
nba_player_aggregated_game_stats_historical_collection = db['nba_player_aggregated_game_stats_historical']
cached_web_api_response_collection = db['cached_web_api_response']

Pinged your deployment. You successfully connected to MongoDB!


In [196]:
############# SPORTS BETTING SANDBOX API ################

#########################################################
# get_event_odds ########################################
def get_event_odds(sports):
    url = 'https://sportsbettingsandboxapi.com/odds-api/events/get'
    response = requests.post(url, json={ 'sports': sports }).json()['data']
    return response
#########################################################

#########################################################
# get_event_odds ########################################
#[derive(Debug, Deserialize, Clone)]
#[serde(rename_all = "camelCase")]
# pub struct GetOddsRequest {
#     pub sports: OddsApiSports,
#     pub regions: OddsApiRegions,
#     pub markets: Vec<String>,
#     pub odds_format: OddsFormat,
#     pub bookmakers: Vec<Bookmakers>
# }
def get_odds(req):
    url = 'https://sportsbettingsandboxapi.com/odds-api/odds/get'
    response = requests.post(url, json=req).json()['data']['events']
    return response
#########################################################

In [197]:
##################### ML FUNCS ##########################

#########################################################
# get_nba_player_season_stats_long_shot_avgs ##############################
def get_nba_player_season_stats_long_shot_avgs(season, num_games):
    player_stats_for_clustering = ["points", "assists", "totReb", "fgm", "fga", 
    "tpm", "tpa", "ftm", "fta", "turnovers", "blocks", "steals"]
    
    game_stats_per_player = dict()
    player_objs = list(nba_player_aggregated_game_stats_historical_collection.find({ 'season': season, 'seasonType': 'ALL' }))
    
    for player in player_objs:
        if player['playerId'] in game_stats_per_player:
            game_stats_per_player[player['playerId']] = game_stats_per_player[player['playerId']] + list(player['playerStats'].values())
        else:
            game_stats_per_player[player['playerId']] = list(player['playerStats'].values())

    all_players_avg_stats = []
    for player_id, game_stats in game_stats_per_player.items():
        player_games_stats = pd.DataFrame(game_stats)
        player_games_stats = player_games_stats[player_games_stats['min'] > 0]

        for stat in player_stats_for_clustering:
            player_games_stats[stat] = player_games_stats[stat] * (36 / player_games_stats['min']) 
        
        player_games_stats = player_games_stats.sort_values(by="dateStart", ascending=False)[player_stats_for_clustering]
        player_avg_stats = player_games_stats.mean()
        player_avg_stats['playerId'] = player_id
        all_players_avg_stats.append(player_avg_stats)

    all_players_avg_stats = pd.DataFrame(all_players_avg_stats)

    # FIND ROOT CAUSE OF NA PROBLEM
    print(all_players_avg_stats.isnull().sum())  # Shows count of NaNs per column

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(all_players_avg_stats.drop("playerId", axis=1))

    kmeans = KMeans(n_clusters=5, random_state=42)
    all_players_avg_stats["role_cluster"] = kmeans.fit_predict(X_scaled)
    print(all_players_avg_stats)
        
        # short_player_stats_df = long_player_stats_df.head(num_games)[player_stats_for_clustering]
        
#########################################################

In [198]:
get_nba_player_season_stats_long_shot_avgs(2024, 10)

points       3
assists      3
totReb       3
fgm          3
fga          3
tpm          3
tpa          3
ftm          3
fta          3
turnovers    3
blocks       3
steals       3
playerId     0
dtype: int64


ValueError: Input X contains NaN.
KMeans does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values